# Pipeline runner
One notebook runs the whole pipeline: set the widgets, then **Run all**.

| widget | meaning |
|---|---|
| `load_type` | **incremental**: WFP is skipped if the file did not change, and only PBS weeks that are not in Silver yet are processed. **full_backfill**: reload everything selected (safe to repeat: no duplicates). |
| `target_dates` | Weeks to run, e.g. `24.09.2026,01.10.2026`. Blank = every week found in the volume folder. |
| `start_date`, `end_date` | Optional window for a **backfill** of an older period (dd.mm.yyyy). |
| `process_wfp`, `process_pbs`, `process_gold` | Switch one part off (Gold refreshes the star schema after Silver). |
| `wfp_countries`, `lookback_months` | Countries kept in Silver WFP, and how many months back an incremental load re-checks. |
| `reset_tables` | `yes` drops and recreates all tables (use only when a table layout changed). |
| `run_validation` | Run the `validation` checks at the end. |

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full_backfill"])
dbutils.widgets.text("target_dates", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("base_dir", "/Volumes/workspace/default/spi_data/samples")
dbutils.widgets.text("wfp_csv_path", "/Volumes/workspace/default/spi_data/samples/wfpvam_foodprices.csv")
dbutils.widgets.dropdown("process_wfp", "yes", ["yes", "no"])
dbutils.widgets.dropdown("process_pbs", "yes", ["yes", "no"])
dbutils.widgets.dropdown("process_gold", "yes", ["yes", "no"])
dbutils.widgets.text("wfp_countries", "Pakistan")
dbutils.widgets.text("lookback_months", "3")
dbutils.widgets.dropdown("reset_tables", "no", ["no", "yes"])
dbutils.widgets.dropdown("run_validation", "yes", ["yes", "no"])

In [0]:
# ---- settings and small helpers (the same block is in every notebook) ----
import json, os, re, uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

CATALOG = "workspace"      # change these two lines if your catalog / schema have other names
SCHEMA = "default"

def T(name):
    """Full table name, e.g. T("silver_quintile_index") -> workspace.default.silver_quintile_index"""
    return f"{CATALOG}.{SCHEMA}.{name}"

AUDIT_TABLE = T("pipeline_execution_logs")
QUARANTINE_TABLE = T("quarantine_records")

def now():
    """Current time (UTC)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)

def sql_text(text):
    """Makes a value safe inside a SQL string: O'Brien -> O''Brien"""
    return str(text).replace("'", "''")

def norm_date(text):
    """Turns 1.10.2026, 01-10-2026 or 2026-10-01 into 01.10.2026"""
    for pattern in ["%d.%m.%Y", "%d-%m-%Y", "%Y-%m-%d"]:
        try:
            return datetime.strptime(str(text).strip(), pattern).strftime("%d.%m.%Y")
        except ValueError:
            pass
    raise ValueError(f"Cannot read the date '{text}'. Use dd.mm.yyyy, for example 01.10.2026")

# ---- execution log: one row per layer, per file/week, per run ----
AUDIT_COLUMNS = ("log_id string, batch_id string, load_type string, layer string, file_processed string, "
                 "target_table string, start_time timestamp, end_time timestamp, status string, "
                 "rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, "
                 "error_message string, message string")

def log_audit(batch_id, load_type, layer, file_processed, target_table, start_time, status,
              rows_read=0, rows_inserted=0, rows_updated=0, rows_quarantined=0, message=""):
    """status is Success, Warning (success with notes), Skipped or Failure."""
    message = (message or "")[:1500]
    error_message = message if status == "Failure" else None      # error_message is only filled for failures
    row = (str(uuid.uuid4()), batch_id, load_type, layer, str(file_processed), target_table,
           start_time, now(), status, int(rows_read), int(rows_inserted), int(rows_updated),
           int(rows_quarantined), error_message, message)
    spark.createDataFrame([row], AUDIT_COLUMNS).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# ---- quarantine: rows (or whole files) that failed a check, saved together with the reason ----
def quarantine_from_list(items):
    """items = list of (record_ref, reason, record_json)"""
    return spark.createDataFrame(items, "record_ref string, reason string, record_json string")

def save_quarantine(df, layer, source_key, batch_id):
    """Replaces the earlier quarantine rows of the same layer and file, so re-runs never pile up duplicates.
    df needs the columns record_ref, reason and record_json."""
    spark.sql(f"DELETE FROM {QUARANTINE_TABLE} WHERE layer = '{layer}' AND source_file = '{sql_text(source_key)}'")
    rows = df.select(F.expr("uuid()").alias("quarantine_id"), F.lit(batch_id).alias("batch_id"),
                     F.lit(layer).alias("layer"), F.lit(source_key).alias("source_file"),
                     "record_ref", "reason", "record_json", F.current_timestamp().alias("quarantined_at"))
    rows.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)


In [0]:
import time

load_type = dbutils.widgets.get("load_type")
base_dir = dbutils.widgets.get("base_dir").rstrip("/")
batch_id = f"{load_type}_{now():%Y%m%dT%H%M%S}"          # one id for everything this run does
print("batch_id:", batch_id)

errors = []                                                # collected so the run can finish and then report

def run_notebook(title, notebook, timeout, parameters):
    """Runs a child notebook. Waits and retries when the Free Edition concurrency limit is hit."""
    print(f"--- {title} ---")
    for attempt in range(3):
        try:
            output = dbutils.notebook.run(notebook, timeout, parameters)
            print("   ->", output[:300] if output else "done")
            return True
        except Exception as e:
            if "REQUEST_LIMIT_EXCEEDED" in str(e) and attempt < 2:
                print("   too many runs at once, waiting 60 seconds...")
                time.sleep(60)
            else:
                print("   FAILED:", str(e)[:400])
                errors.append(f"{title}: {str(e)[:300]}")
                return False

In [0]:
# Step 0: create the tables (and reset them if asked)
run_notebook("Step 0: Schemas", "Schemas", 600, {"reset_tables": dbutils.widgets.get("reset_tables")})

In [0]:
# Step 1: WFP. Silver only runs if Bronze worked.
if dbutils.widgets.get("process_wfp") == "yes":
    bronze_ok = run_notebook("Step 1a: Bronze WFP", "bronzewfp", 1800,
                             {"wfp_file_path": dbutils.widgets.get("wfp_csv_path"),
                              "load_type": load_type, "batch_id": batch_id})
    if bronze_ok:
        run_notebook("Step 1b: Silver WFP", "silverwfp", 1800,
                     {"load_type": load_type, "batch_id": batch_id,
                      "countries": dbutils.widgets.get("wfp_countries"),
                      "lookback_months": dbutils.widgets.get("lookback_months")})
else:
    print("WFP skipped")

In [0]:
# Step 2: decide WHICH PBS weeks to run
REPORT_NAME = re.compile(r"^(?:3\.-SPI-Report|SPI_Report)[-_](\d{1,2}[.\-]\d{1,2}[.\-]\d{4})\.xlsx$", re.I)
ANNEX_NAME = re.compile(r"^(?:Annex|SPI_Annexure|SPI_Annex)[-_](\d{1,2}[.\-]\d{1,2}[.\-]\d{4})\.xlsx$", re.I)

def weeks_in_volume(folder):
    """Weeks that have BOTH a report file and an annexure file."""
    reports, annexes = set(), set()
    for f in dbutils.fs.ls(folder):
        if REPORT_NAME.match(f.name):
            reports.add(norm_date(REPORT_NAME.match(f.name).group(1)))
        if ANNEX_NAME.match(f.name):
            annexes.add(norm_date(ANNEX_NAME.match(f.name).group(1)))
    print("weeks with only one of the two files (ignored):", sorted(reports ^ annexes))
    return reports & annexes

def weeks_in_silver():
    """Weeks that are already in all three PBS Silver tables."""
    done = None
    for name in ["silver_quintile_index", "silver_item_weight_impact", "silver_city_price_observation"]:
        if not spark.catalog.tableExists(T(name)):
            return set()
        rows = spark.table(T(name)).select(F.date_format("week_ending_date", "dd.MM.yyyy")).distinct().collect()
        weeks = set(r[0] for r in rows)
        done = weeks if done is None else done & weeks
    return done

def as_date(week):
    return datetime.strptime(week, "%d.%m.%Y")

weeks_to_run = []
if dbutils.widgets.get("process_pbs") == "yes":
    typed_weeks = [norm_date(w) for w in dbutils.widgets.get("target_dates").split(",") if w.strip()]
    if typed_weeks:
        selected = typed_weeks                                  # exactly what you typed
    else:
        selected = list(weeks_in_volume(base_dir))              # everything found in the folder
        if dbutils.widgets.get("start_date").strip():           # backfill window (optional)
            start = as_date(norm_date(dbutils.widgets.get("start_date")))
            selected = [w for w in selected if as_date(w) >= start]
        if dbutils.widgets.get("end_date").strip():
            end = as_date(norm_date(dbutils.widgets.get("end_date")))
            selected = [w for w in selected if as_date(w) <= end]
        if load_type == "incremental":                          # skip weeks that are already loaded
            already_loaded = weeks_in_silver()
            selected = [w for w in selected if w not in already_loaded]
    weeks_to_run = sorted(set(selected), key=as_date)
    print(f"load_type = {load_type}; PBS weeks to process ({len(weeks_to_run)}): {weeks_to_run}")
    if len(weeks_to_run) == 0:
        print("Nothing to do for PBS. Use full_backfill or target_dates to run a week again.")

In [0]:
# Step 3: PBS. All selected weeks go through each notebook in one go; every week is logged separately.
if len(weeks_to_run) > 0:
    dates = ",".join(weeks_to_run)
    run_notebook("Step 3a: Bronze PBS", "bronzepbs", 3000,
                 {"base_dir": base_dir, "week_ending_dates": dates, "load_type": load_type, "batch_id": batch_id})
    run_notebook("Step 3b: Silver PBS", "silverpbs", 3000,
                 {"week_ending_dates": dates, "load_type": load_type, "batch_id": batch_id})

In [0]:
# Step 3c: Gold. Rebuilds the star schema from Silver, in the same batch.
if dbutils.widgets.get("process_gold") == "yes":
    if len(weeks_to_run) > 0:
        gold_weeks = ",".join(weeks_to_run)                  # refresh the facts of the weeks we just loaded
    elif load_type == "full_backfill" and dbutils.widgets.get("target_dates").strip() == "":
        gold_weeks = "all"
    else:
        gold_weeks = "none"                                  # no new PBS weeks (the WFP fact is still refreshed)
    run_notebook("Step 3c: Gold", "gold", 1800,
                 {"load_type": load_type, "batch_id": batch_id, "week_ending_dates": gold_weeks,
                  "lookback_months": dbutils.widgets.get("lookback_months")})

In [0]:
# Step 4: checks
if dbutils.widgets.get("run_validation") == "yes":
    run_notebook("Step 4: Validation", "validation", 1200, {"fail_on_error": "yes", "batch_id": batch_id})

In [0]:
# Summary: what this run did
display(spark.sql(f"""
    SELECT layer, file_processed, status, rows_read, rows_inserted, rows_updated, rows_quarantined, start_time, message, error_message
    FROM {AUDIT_TABLE} WHERE batch_id = '{batch_id}' ORDER BY start_time"""))

if len(errors) > 0:
    raise Exception("The pipeline finished with errors:\n - " + "\n - ".join(errors))
print("Pipeline completed without errors. batch_id =", batch_id)